<!--nav--> [🗺 Learning path](README.md) · **46/49** · ◀ [Anatomy of a Decode Step](./Anatomy_Of_A_Decode_Step.ipynb) · [The Optimization Stack](./The_Optimization_Stack.ipynb) ▶

# Attention Kernels From Scratch: Online Softmax, Tiling, and Why FlashAttention Works

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Attention_Kernels_From_Scratch.ipynb)

Every notebook so far treated attention as a box with a cost. This one opens it.

FlashAttention is usually explained as "it's faster because it's IO-aware," which is true and
useless. The actual mechanism is a **beautiful piece of numerical algebra** — you can compute softmax
without ever materializing the thing you're taking the softmax of — and once you see it, paged
attention, FlashDecoding and the whole KV-cache design stop being arbitrary.

You will **implement it and verify it is exact**, in numpy, on a CPU.

| Part | What you'll do |
|---|---|
| **1** | The N² problem: what naive attention actually allocates |
| **2** | **Online softmax** — derive it, implement it, prove it equals the reference to machine precision |
| **3** | **FlashAttention forward from scratch**, verified against naive attention |
| **4** | Why tiling wins: the IO-complexity argument, with numbers |
| **5** | **FlashDecoding**: the batch-1 long-context problem and the split-K fix |
| **6** | **PagedAttention** as a kernel: block tables, and what changes inside the loop |
| **7** | Why FlashAttention-3 is Hopper-specific, and what that means for portability |

**Runs on:** any CPU. numpy only — the point is that the *algorithm* is the insight, not the CUDA.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · The N² problem

Standard attention, written the way the math is usually presented:

```
S = Q Kᵀ / √d        [N × N]   ← this is the problem
P = softmax(S)       [N × N]   ← and so is this
O = P V              [N × d]
```

For a 32k-token sequence, `S` is 32768² floats = **4 GB per head, per layer**. With 32 heads that is
128 GB of intermediate storage for one layer of one sequence. Obviously nobody does this — but it's
worth seeing *why* the naive form is impossible before seeing what replaces it.

The subtlety: it isn't the FLOPs that hurt. It's that `S` has to travel **to HBM and back** twice
(write it, read it for softmax, write it, read it for the `PV` matmul). Attention is
**memory-movement bound**, and the movement is of a matrix that doesn't need to exist.

In [ ]:
import numpy as np

def naive_attention(Q, K, V):
    '''The textbook form. Materializes the full N x N score matrix.'''
    d = Q.shape[-1]
    S = Q @ K.T / np.sqrt(d)              # [N, N]  <- the 4 GB problem
    P = np.exp(S - S.max(axis=-1, keepdims=True))
    P = P / P.sum(axis=-1, keepdims=True)
    return P @ V, S.nbytes

for n in (1024, 8192, 32768, 131072):
    bytes_S = n * n * 2                    # fp16
    print(f"seq {n:>7,}: S matrix = {bytes_S/1e9:>8.2f} GB per head"
          f"   x32 heads = {bytes_S*32/1e9:>9.1f} GB")

print("\nAnd the FLOPs, for comparison:")
for n in (1024, 8192, 32768):
    flops = 2 * 2 * n * n * 128            # QK^T and PV, head_dim 128
    print(f"  seq {n:>6,}: {flops/1e9:>8.1f} GFLOP per head "
          f"({flops/1e9/312:>6.3f} ms on an A100 at peak)")

print("\nThe compute is affordable; the intermediate is not. That gap is the whole story -")
print("and it is the same 'memory movement is the bottleneck' theme as serving-fundamentals, one")
print("level down.")

## Part 2 · Online softmax: computing softmax without the vector

The obstacle looks fundamental: softmax needs `max` and `sum` over the **whole** row before it can
normalize anything. How can you process a row in chunks if the normalizer depends on chunks you
haven't seen?

The trick is to keep a **running max** and a **running sum**, and *rescale* what you've accumulated
whenever the max changes.

Suppose you've processed a block and hold `m` (running max) and `ℓ` (running sum of `exp(x−m)`). A
new block arrives with max `m'`. The combined max is `m_new = max(m, m')`. Every previously
accumulated exponential was computed relative to the *old* `m`, so it must be corrected:

$$\ell_{new} = \ell \cdot e^{m - m_{new}} + \sum_{x \in \text{block}} e^{x - m_{new}}$$

That correction factor `e^{m − m_new}` is the entire idea. It is exact — no approximation — and it
also applies to the accumulated **output**, which is what makes the whole attention computable in
one streaming pass.

Let's implement it and check it against the reference:

In [ ]:
def softmax_reference(x):
    e = np.exp(x - x.max())
    return e / e.sum()

def softmax_online(x, block_size=7):
    '''Streaming softmax: never sees more than `block_size` elements at once.'''
    m = -np.inf          # running max
    l = 0.0              # running sum of exp(x - m)
    for i in range(0, len(x), block_size):
        block = x[i:i + block_size]
        m_block = block.max()
        m_new = max(m, m_block)
        # rescale the old accumulator to the new max, then add this block's contribution
        l = l * np.exp(m - m_new) + np.exp(block - m_new).sum()
        m = m_new
    # a second pass only to emit the normalized values (the fused kernel avoids even this)
    return np.exp(x - m) / l

rng = np.random.default_rng(0)
for scale, label in [(1.0, "well-behaved"), (50.0, "large values (overflow bait)"),
                     (1e-6, "tiny values")]:
    x = rng.standard_normal(1000) * scale
    ref, onl = softmax_reference(x), softmax_online(x)
    err = np.abs(ref - onl).max()
    print(f"{label:<30} max abs error = {err:.3e}   "
          f"{'✅ exact to machine precision' if err < 1e-15 else '⚠ check'}")

# Block size must not matter - that is the point.
x = rng.standard_normal(1000) * 10
errs = [np.abs(softmax_reference(x) - softmax_online(x, b)).max() for b in (1, 3, 16, 128, 1000)]
print(f"\nblock sizes 1..1000 all agree with the reference: max error {max(errs):.3e}")
print("Online softmax is not an approximation. It is the same number, computed in a different order.")

### The same trick, applied to the output

Softmax alone isn't enough — we need `O = P V` without materializing `P`. The same rescaling works,
because the output accumulator is a weighted sum whose weights all share the normalizer:

$$O_{new} = O \cdot \frac{\ell \, e^{m - m_{new}}}{\ell_{new}} + \frac{\sum e^{x - m_{new}} V}{\ell_{new}}$$

In practice kernels keep the *unnormalized* accumulator and divide once at the end — fewer
operations, same result. That's what the implementation below does.

## Part 3 · FlashAttention forward, from scratch

Now the whole thing. Two nested loops over blocks; **`S` is never materialized** — only a
`[block_q × block_k]` tile ever exists, which fits in on-chip SRAM.

In [ ]:
def flash_attention(Q, K, V, block_q=32, block_k=32):
    '''FlashAttention forward pass. Peak extra memory is O(block_q x block_k), not O(N^2).'''
    N, d = Q.shape
    O = np.zeros((N, d), dtype=np.float64)
    m = np.full(N, -np.inf)                  # running max per query row
    l = np.zeros(N)                          # running sum per query row
    scale = 1.0 / np.sqrt(d)
    max_tile = 0

    for i in range(0, N, block_q):                       # outer loop over query blocks
        qi = slice(i, min(i + block_q, N))
        Qi = Q[qi]
        Oi = np.zeros((Qi.shape[0], d))
        mi = np.full(Qi.shape[0], -np.inf)
        li = np.zeros(Qi.shape[0])

        for j in range(0, N, block_k):                   # inner loop over key blocks
            kj = slice(j, min(j + block_k, N))
            Sij = (Qi @ K[kj].T) * scale                 # the ONLY materialized tile
            max_tile = max(max_tile, Sij.nbytes)

            m_block = Sij.max(axis=1)
            m_new = np.maximum(mi, m_block)
            correction = np.exp(mi - m_new)              # rescale factor for what we already have
            P = np.exp(Sij - m_new[:, None])

            li = li * correction + P.sum(axis=1)
            Oi = Oi * correction[:, None] + P @ V[kj]    # accumulate UNNORMALIZED output
            mi = m_new

        O[qi] = Oi / li[:, None]                         # normalize once, at the end
        m[qi], l[qi] = mi, li
    return O, max_tile

rng = np.random.default_rng(1)
N, d = 512, 64
Q = rng.standard_normal((N, d)); K = rng.standard_normal((N, d)); V = rng.standard_normal((N, d))

ref, s_bytes = naive_attention(Q, K, V)
for bq, bk in [(32, 32), (64, 128), (128, 64), (1, 1), (N, N)]:
    out, tile_bytes = flash_attention(Q, K, V, bq, bk)
    err = np.abs(ref - out).max()
    print(f"block_q={bq:>4} block_k={bk:>4}   max error vs naive = {err:.3e}   "
          f"peak tile {tile_bytes/1024:>7.1f} KB  (naive S: {s_bytes/1024:.0f} KB)")

print("\nEvery block configuration produces the SAME answer as naive attention, to machine")
print("precision, while the largest intermediate shrinks from the full N x N matrix to one tile.")
print("That is FlashAttention. Everything else is engineering to make the tiles land in SRAM.")

### Watch the tiles sweep

The loop above is easier to believe once you see it. Below, the grid is the `N × N` score matrix
that FlashAttention **never builds**. Each square is one `block_q × block_k` tile; the sweep shows
the order the kernel visits them, and only the highlighted tile exists in fast memory at any moment.

The bars on the right are the running state for the four query rows in the current row-block:
`m` (the running max) and `ℓ` (the running sum). Watch what happens when a tile contains a larger
score than anything seen so far — **`m` jumps, and the accumulator behind it is rescaled**. That
rescaling is the entire algorithm; everything else is bookkeeping.

In [ ]:
# Re-run the kernel, recording the state after every tile so the animation shows real numbers.
def flash_trace(Q, K, V, block_q, block_k):
    N, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    steps = []
    for i in range(0, N, block_q):
        qi = slice(i, min(i + block_q, N))
        Qi = Q[qi]
        mi = np.full(Qi.shape[0], -np.inf)
        li = np.zeros(Qi.shape[0])
        for j in range(0, N, block_k):
            kj = slice(j, min(j + block_k, N))
            Sij = (Qi @ K[kj].T) * scale
            m_new = np.maximum(mi, Sij.max(axis=1))
            rescaled = bool(np.any(m_new > mi + 1e-12) and np.isfinite(mi).any())
            correction = np.exp(mi - m_new)
            P = np.exp(Sij - m_new[:, None])
            li = li * correction + P.sum(axis=1)
            mi = m_new
            steps.append(dict(qb=i // block_q, kb=j // block_k, rescaled=rescaled,
                              m=[round(float(v), 3) for v in mi[:4]],
                              l=[round(float(v), 3) for v in li[:4]]))
    return steps

NV, DV, BQ, BK = 256, 64, 32, 32
rngv = np.random.default_rng(7)
Qv = rngv.standard_normal((NV, DV)); Kv = rngv.standard_normal((NV, DV)); Vv = rngv.standard_normal((NV, DV))
trace = flash_trace(Qv, Kv, Vv, BQ, BK)
n_side = NV // BQ

viz = dict(steps=trace, n=n_side, block=BQ,
           tile_kb=round(BQ * BK * 8 / 1024, 1), full_kb=round(NV * NV * 8 / 1024, 1))
print(f"{len(trace)} tiles, {n_side}x{n_side} grid; "
      f"{sum(s['rescaled'] for s in trace)} of them forced a rescale of the accumulator")

show_d3("""
const S = data.steps, n = data.n;
const cell = Math.min(46, Math.floor((W * 0.52) / n)), pad = 3;
const gw = n * (cell + pad), left = 10, top = 44;

root.append("div").attr("id","hdr").style("font","13px system-ui")
    .style("margin-bottom","6px");
const svg = root.append("svg").attr("width", W).attr("height", H);

svg.append("text").attr("x", left).attr("y", 20).attr("font-size", 12)
   .attr("fill", "#555").text("the N x N score matrix FlashAttention never builds");

const g = svg.append("g").attr("transform", `translate(${left},${top})`);
const rects = [];
for (let r = 0; r < n; r++) for (let c = 0; c < n; c++) {
  rects.push(g.append("rect").attr("x", c*(cell+pad)).attr("y", r*(cell+pad))
     .attr("width", cell).attr("height", cell).attr("rx", 3)
     .attr("fill", "#eef0f4").attr("stroke", "#d5d9e0"));
}
svg.append("text").attr("x", left).attr("y", top + n*(cell+pad) + 20).attr("font-size", 11)
   .attr("fill", "#777").text("rows = query blocks, columns = key blocks");

// running-state panel
const px = left + gw + 34;
svg.append("text").attr("x", px).attr("y", 20).attr("font-size", 12).attr("fill", "#555")
   .text("running m and l for the first 4 rows of the active query block");
const bars = [];
for (let k = 0; k < 4; k++) {
  const y = top + k * 46;
  svg.append("text").attr("x", px).attr("y", y + 12).attr("font-size", 11)
     .attr("fill", "#888").text("row " + k);
  bars.push({
    m: svg.append("rect").attr("x", px + 44).attr("y", y).attr("height", 14).attr("rx", 3)
          .attr("fill", "#4338ca").attr("width", 0),
    l: svg.append("rect").attr("x", px + 44).attr("y", y + 18).attr("height", 14).attr("rx", 3)
          .attr("fill", "#0ea5e9").attr("width", 0),
    mt: svg.append("text").attr("x", px + 44).attr("y", y + 12).attr("font-size", 10)
           .attr("fill", "#333"),
    lt: svg.append("text").attr("x", px + 44).attr("y", y + 30).attr("font-size", 10)
           .attr("fill", "#333"),
  });
}
const maxW = Math.max(60, W - px - 60);
const allM = S.flatMap(t => t.m), allL = S.flatMap(t => t.l);
const mlo = Math.min(...allM), mhi = Math.max(...allM) + 1e-9, lmax = Math.max(...allL);

// Keep the value legible whether the bar is short or nearly full width.
function label(sel, text, barW) {
  const inside = barW > maxW - 62;
  sel.text(text)
     .attr("x", px + 44 + barW + (inside ? -6 : 6))
     .attr("text-anchor", inside ? "end" : "start")
     .attr("fill", inside ? "#fff" : "#333");
}

const ctl = root.append("div").style("margin-top","8px").style("font","13px system-ui");
const btn = ctl.append("button").text("⏸ pause").style("margin-right","10px");
const slider = ctl.append("input").attr("type","range").attr("min",0)
   .attr("max", S.length - 1).attr("value", 0).style("width","300px").style("vertical-align","middle");

function draw(i) {
  const s = S[i];
  rects.forEach((r, idx) => {
    const rr = Math.floor(idx / n), cc = idx % n;
    const done = rr < s.qb || (rr === s.qb && cc < s.kb);
    const here = rr === s.qb && cc === s.kb;
    r.attr("fill", here ? (s.rescaled ? "#f59e0b" : "#4338ca") : done ? "#c7d2fe" : "#eef0f4")
     .attr("stroke", here ? "#111" : "#d5d9e0").attr("stroke-width", here ? 2 : 1);
  });
  s.m.forEach((v, k) => {
    const wm = Math.max(2, maxW * (v - mlo) / (mhi - mlo)), wl = Math.max(2, maxW * s.l[k] / lmax);
    bars[k].m.attr("width", wm);
    bars[k].l.attr("width", wl);
    label(bars[k].mt, "m = " + v.toFixed(2), wm);
    label(bars[k].lt, "l = " + s.l[k].toFixed(2), wl);
  });
  root.select("#hdr").html(
    `tile <b>${i + 1}/${S.length}</b> &nbsp;·&nbsp; in fast memory: <b>${data.tile_kb} KB</b> ` +
    `(the full matrix would be ${data.full_kb} KB) &nbsp;·&nbsp; ` +
    (s.rescaled ? `<span style="color:#b45309"><b>new max → accumulator rescaled</b></span>`
                : `<span style="color:#666">max unchanged</span>`));
  slider.property("value", i);
}

let i = 0, playing = true;
draw(0);
const timer = setInterval(() => { if (playing) { i = (i + 1) % S.length; draw(i); } }, 320);
btn.on("click", () => { playing = !playing; btn.text(playing ? "⏸ pause" : "▶ play"); });
slider.on("input", function () { playing = false; btn.text("▶ play"); i = +this.value; draw(i); });
""", viz, height=max(320, 44 + (256 // 32) * 49 + 40))

## Part 4 · Why tiling wins: the IO argument

Now the performance claim, made precisely. Let `M` be the size of fast on-chip memory (SRAM/shared
memory, ~100–200 KB per SM). Counting **HBM accesses**:

| | HBM traffic |
|---|---|
| **Naive** | `O(N² + N·d)` — the score matrix goes out and comes back |
| **Flash** | `O(N²·d² / M)` — each block of K,V is read once per query block |

Both are quadratic in `N`, so the ratio is a **constant factor**, not something that improves as
sequences grow: roughly `M / d²`, set by how many K,V columns fit in fast memory relative to the
head dimension. Bigger SRAM helps; bigger heads hurt. The FLOPs are *identical* either way —
FlashAttention is not a cheaper algorithm, it is the same algorithm that moves less data.

What *does* grow with `N` is the thing that actually blocked long context: naive attention's **peak
memory** is `O(N²)` — 34 GB per head at 128k — while Flash's is `O(block_q × block_k)`, a few KB.
A constant-factor traffic win plus an asymptotic *capacity* win.

This is exactly [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb)'s roofline logic applied one level down: attention was memory-bound, so
the win came from moving less, not computing less.

In [ ]:
def hbm_traffic(N, d, M_bytes=100_000, dtype_bytes=2):
    # Naive: write S, read S (softmax), write P, read P (PV) - plus Q,K,V,O
    naive = (4 * N * N + 4 * N * d) * dtype_bytes
    # Flash: K,V read once per query block; Q,O read/written once
    block = max(1, M_bytes // (4 * d * dtype_bytes))
    n_blocks = max(1, int(np.ceil(N / block)))
    flash = (2 * N * d * n_blocks + 2 * N * d) * dtype_bytes
    return naive, flash, block

D = 128
print(f"{'seq len':>9}{'naive HBM':>14}{'flash HBM':>14}{'reduction':>12}{'peak naive S':>14}")
print("-" * 64)
for N in (512, 2048, 8192, 32768, 131072):
    naive, flash, block = hbm_traffic(N, D)
    print(f"{N:>9,}{naive/1e9:>12.3f}GB{flash/1e9:>12.3f}GB{naive/flash:>11.1f}x"
          f"{N * N * 2 / 1e9:>12.2f}GB")

# The ratio the model should reproduce, derived by hand: 4N^2 / (2*N*d*(N/block)) = 2*block/d,
# and block = M / (4*d*bytes), so the whole thing collapses to M / (2*d^2*bytes) - no N in it.
predicted = 100_000 / (2 * D ** 2 * 2)
measured = [hbm_traffic(N, D)[0] / hbm_traffic(N, D)[1] for N in (8192, 32768, 131072)]
assert all(abs(r - predicted) < 0.05 for r in measured), (predicted, measured)

print(f"\nThe reduction is FLAT at ~{predicted:.1f}x, and the arithmetic says it must be:")
print("both traffic terms are quadratic in N, so N cancels and the ratio is M / (2*d^2*bytes).")
print("Halve the head dim and it quadruples; double the SRAM and it doubles. Sequence length")
print("does not appear.")
print("\nThe column that DOES blow up is the last one - the naive score matrix itself, O(N^2)")
print("per head. 34 GB at 128k, against a few KB of tile for flash. That capacity wall, not")
print("the traffic ratio, is what made long context impossible before tiling.")
print("\nNote what is NOT in this table: FLOPs. They are unchanged. FlashAttention is a pure")
print("data-movement optimization, which is the same category as paging the KV cache (vLLM).")

## Part 5 · FlashDecoding: the batch-1 problem

FlashAttention parallelizes over **query blocks**. During *decode* there is exactly **one query
token** — so there is only one query block, and most of the GPU sits idle while a single thread
block walks a 128k-token KV cache sequentially.

The fix, **FlashDecoding**, is to split the *keys* instead: partition the KV cache into chunks,
attend to each chunk in parallel, then combine the partial results — using the **same rescaling
identity** from Part 2, now applied across chunks rather than within a row.

In [ ]:
def flash_decoding(q, K, V, n_splits=4):
    '''Decode attention for ONE query, parallelized by splitting the KV cache.'''
    d = q.shape[-1]
    scale = 1.0 / np.sqrt(d)
    N = K.shape[0]
    chunk = int(np.ceil(N / n_splits))

    partials = []                                    # each "worker" handles one KV chunk
    for s in range(n_splits):
        sl = slice(s * chunk, min((s + 1) * chunk, N))
        if sl.start >= N:
            continue
        S = (q @ K[sl].T) * scale
        m = S.max()
        p = np.exp(S - m)
        partials.append({"m": m, "l": p.sum(), "o": p @ V[sl]})

    # combine with the SAME identity as Part 2, across workers instead of within a row
    m_all = max(p["m"] for p in partials)
    l_all = sum(p["l"] * np.exp(p["m"] - m_all) for p in partials)
    o_all = sum(p["o"] * np.exp(p["m"] - m_all) for p in partials)
    return o_all / l_all

rng = np.random.default_rng(2)
N, d = 4096, 64
q = rng.standard_normal(d); K = rng.standard_normal((N, d)); V = rng.standard_normal((N, d))

S = (q @ K.T) / np.sqrt(d)
p = np.exp(S - S.max()); ref = (p / p.sum()) @ V

print(f"{'splits':>8}{'max error vs reference':>26}{'parallel workers':>19}")
print("-" * 54)
for n in (1, 2, 4, 8, 16, 64):
    out = flash_decoding(q, K, V, n)
    print(f"{n:>8}{np.abs(ref - out).max():>26.3e}{n:>19}")

print("\nExact for every split count - the combination step is algebra, not approximation.")
print("\nWhy this matters (decode-anatomy's decode-step budget): at batch 1 with a long context, the")
print("attention slice is a serial walk over the KV cache. Splitting it across 16 workers")
print("turns an idle GPU into a busy one WITHOUT changing the answer. This is the single")
print("most important kernel-level optimization for long-context INTERACTIVE serving.")

## Part 6 · PagedAttention as a kernel

[vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) described PagedAttention as virtual memory for the KV cache. Inside the kernel, the
change is small and specific: instead of walking a contiguous `[N, d]` array, the loop consults a
**block table** and gathers each block from wherever it lives.

```
 contiguous:   for j in range(0, N, BLOCK):  K_block = K[j : j+BLOCK]
 paged:        for b in block_table[seq]:    K_block = K_pool[b]      # indirection
```

Everything else — the online softmax, the rescaling, the accumulation — is **identical**. That's why
PagedAttention composes with FlashAttention rather than competing with it.

The cost is one extra indirection per block, which is negligible because blocks are 16+ tokens. The
benefit is everything [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) measured.

In [ ]:
def paged_flash_decoding(q, k_pool, v_pool, block_table, block_size=16):
    '''Decode attention over a KV cache scattered across a block pool.'''
    d = q.shape[-1]
    scale = 1.0 / np.sqrt(d)
    m, l, o = -np.inf, 0.0, np.zeros(d)
    for blk in block_table:                       # <- the ONLY structural difference
        K_b = k_pool[blk]                          # gather, rather than slice
        V_b = v_pool[blk]
        S = (q @ K_b.T) * scale
        m_new = max(m, S.max())
        corr = np.exp(m - m_new)
        p = np.exp(S - m_new)
        l = l * corr + p.sum()
        o = o * corr + p @ V_b
        m = m_new
    return o / l

rng = np.random.default_rng(3)
N, d, BLK = 512, 64, 16
K = rng.standard_normal((N, d)); V = rng.standard_normal((N, d)); q = rng.standard_normal(d)

# Scatter the sequence's blocks randomly through a larger pool, as a real allocator would.
n_blocks = N // BLK
pool_size = 200
k_pool = rng.standard_normal((pool_size, BLK, d))
v_pool = rng.standard_normal((pool_size, BLK, d))
slots = rng.permutation(pool_size)[:n_blocks]
for i, slot in enumerate(slots):
    k_pool[slot] = K[i * BLK:(i + 1) * BLK]
    v_pool[slot] = V[i * BLK:(i + 1) * BLK]

S = (q @ K.T) / np.sqrt(d); p = np.exp(S - S.max()); ref = (p / p.sum()) @ V
out = paged_flash_decoding(q, k_pool, v_pool, slots, BLK)
print(f"contiguous vs paged (blocks scattered across a {pool_size}-block pool):")
print(f"  max abs error = {np.abs(ref - out).max():.3e}")
print(f"  physical block order actually used: {[int(b) for b in slots[:8]]} ...")
print("\nSame answer, arbitrary physical layout. The kernel does not care where the blocks are -")
print("which is exactly what makes the memory management in vLLM possible.")

## Part 7 · Why FlashAttention-3 is Hopper-specific

FA-2 is portable in spirit — the algorithm above runs anywhere. FA-3 gets its extra speed from
hardware features that **only exist on Hopper**:

| Feature | What it does | Where it exists |
|---|---|---|
| **TMA** (Tensor Memory Accelerator) | async bulk copies HBM↔SRAM without occupying threads | Hopper+ |
| **Warp specialization** | dedicated producer/consumer warps overlapping copy and math | Hopper+ |
| **FP8 tensor cores in the inner loop** | attention math at fp8 | Ada/Hopper+ |
| **Async softmax/GEMM overlap** | softmax of tile *i* while GEMM of tile *i+1* runs | needs the above |

This is [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb)'s thesis in concrete form: **the algorithm ports, the peak performance doesn't.**
On AMD the same algorithm is realized through Composable Kernel / AITER with different primitives —
correct and fast, but a different implementation, tuned for wavefront-64.

**The practical takeaway:** when you benchmark attention on new hardware, you are benchmarking a
*different implementation of the same math*. Expect to re-tune block sizes, and don't assume a
speedup measured on Hopper transfers.

## Recap

1. **Naive attention's problem is the intermediate, not the FLOPs.** `S` is `O(N²)` and it has to
   travel to HBM twice.
2. **Online softmax removes the need for it**, exactly — the running-max rescaling you implemented
   in Part 2 is the entire mechanism, and it's numerically identical to the reference.
3. **FlashAttention is that trick plus tiling.** Same FLOPs, far less data movement — verified
   here to machine precision for every block size.
4. **FlashDecoding applies the same identity across workers**, which is what rescues batch-1
   long-context decode from being a serial walk.
5. **PagedAttention changes one line of the loop** — a gather instead of a slice — which is why it
   composes cleanly with everything above.
6. **FA-3's speed is hardware-specific** even though the algorithm isn't.

### Further reading
- [FlashAttention](https://arxiv.org/abs/2205.14135) · [FlashAttention-2](https://arxiv.org/abs/2307.08691) · [FlashAttention-3](https://arxiv.org/abs/2407.08608)
- [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867) — the 2018 paper the whole thing rests on
- [FlashDecoding](https://pytorch.org/blog/flash-decoding/) · [PagedAttention / vLLM](https://arxiv.org/abs/2309.06180)
- Context in this repo: [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) · [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) · [Anatomy of a Decode Step](./Anatomy_Of_A_Decode_Step.ipynb)